In [7]:
import pandas as pd
import numpy as np



## uploading the data

In [8]:
accidents_files=['C:\\Users\\ThikPad\\Documents\\accidents_project\\2015\\2015.csv' 
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2016\\2016.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2017\\2017.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2018\\2018.csv'
            ,   'C:\\Users\\ThikPad\\Documents\\accidents_project\\2019\\2019.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2020\\2020.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2021\\2021.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2022\\2022.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2023\\2023.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2024\\2024.csv'
            


         ]

carcteristques_files=[
    'C:\\Users\\ThikPad\\Documents\\accidents_project\\2021\\carcteristiques-2021.csv'
    ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2022\\carcteristiques-2022.csv'
    ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2023\\caract-2023.csv'
    ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2024\\caract-2024.csv'
]



usagers_files=['C:\\Users\\ThikPad\\Documents\\accidents_project\\2021\\usagers-2021.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2022\\usagers-2022.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2023\\usagers-2023.csv'
            ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2024\\usagers-2024.csv'
            ]

lieux_files=['C:\\Users\\ThikPad\\Documents\\accidents_project\\2021\\lieux-2021.csv'
             ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2022\\lieux-2022.csv'
             ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2023\\lieux-2023.csv'
            , 'C:\\Users\\ThikPad\\Documents\\accidents_project\\2024\\lieux-2024.csv'
                ]

vehicules_files=['C:\\Users\\ThikPad\\Documents\\accidents_project\\2021\\vehicules-2021.csv'
                ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2022\\vehicules-2022.csv'
                ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2023\\vehicules-2023.csv'
                ,'C:\\Users\\ThikPad\\Documents\\accidents_project\\2024\\vehicules-2024.csv'
]
accidents_df = pd.concat([pd.read_csv(f, sep=';') for f in accidents_files], ignore_index=True)

Usagers_df = pd.concat([pd.read_csv(f, sep=';') for f in usagers_files], ignore_index=True)

# Read Lieux_df without dtype specifications (let pandas infer types)
Lieux_df = pd.concat([pd.read_csv(f, sep=';', dtype=str, low_memory=False) for f in lieux_files], ignore_index=True)

vehicule_df = pd.concat([pd.read_csv(f, sep=';') for f in vehicules_files], ignore_index=True)

carcteristiques_df= pd.concat([pd.read_csv(f, sep=';') for f in carcteristques_files], ignore_index=True)




In [9]:
# Clean and convert the problematic columns after loading
int_columns = ['catr', 'v1', 'circ', 'nbv', 'pr', 'pr1', 'pr2', 'plan', 'larrout', 'surf', 'infra', 'situ', 'vma']

for col in int_columns:
    if col in Lieux_df.columns:
        # Remove parentheses and other non-numeric characters, then convert to int
        Lieux_df[col] = pd.to_numeric(
            Lieux_df[col].astype(str).str.replace(r'[^\d-]', '', regex=True),
            errors='coerce'
        ).astype('Int64')  # Use nullable integer type to handle NaN values

for col in ['an' , 'mois', 'jour', 'hrmn']:
    if col in accidents_df.columns:
        accidents_df[col] = pd.to_numeric(accidents_df[col], errors='coerce').astype('Int64')



In [10]:
accidents_df.info()
accidents_df.head()
accidents_df.isnull().sum()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 901759 entries, 0 to 901758
Data columns (total 9 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Id_accident                         901759 non-null  object 
 1   Lettre Conventionnelle Véhicule     901755 non-null  object 
 2   Année                               901759 non-null  int64  
 3   Lieu Admin Actuel - Territoire Nom  901759 non-null  object 
 4   Type Accident - Libellé             557347 non-null  object 
 5   CNIT                                671697 non-null  object 
 6   Catégorie véhicule                  901759 non-null  object 
 7   Age véhicule                        839554 non-null  float64
 8   Type Accident - Libellé (old)       89416 non-null   object 
dtypes: float64(1), int64(1), object(7)
memory usage: 61.9+ MB


Id_accident                                0
Lettre Conventionnelle Véhicule            4
Année                                      0
Lieu Admin Actuel - Territoire Nom         0
Type Accident - Libellé               344412
CNIT                                  230062
Catégorie véhicule                         0
Age véhicule                           62205
Type Accident - Libellé (old)         812343
dtype: int64

In [ ]:
# Clean the data first - remove rows with missing accident type
accidents_df_cleaned = accidents_df.dropna(subset=['Type Accident - Libellé'], how='any')

print(f"Original dataset shape: {accidents_df.shape}")
print(f"Cleaned dataset shape: {accidents_df_cleaned.shape}")
print(f"Rows removed: {len(accidents_df) - len(accidents_df_cleaned)}")

# Rename Id_accident to Num_Acc for consistency with other datasets
accidents_df_cleaned = accidents_df_cleaned.rename(columns={'Id_accident': 'Num_Acc'})

# Merge accidents with usagers (users/people involved)
# Use left_on and right_on to handle potential column name differences
merged_df = accidents_df_cleaned.merge(Usagers_df, on='Num_Acc', how='left', suffixes=('', '_usagers'))

# Merge with vehicules
merged_df = merged_df.merge(vehicule_df, on='Num_Acc', how='left', suffixes=('', '_vehicule'))

# Merge with lieux (locations)
merged_df = merged_df.merge(Lieux_df, on='Num_Acc', how='left', suffixes=('', '_lieux'))

print(f"\nMerged dataset shape: {merged_df.shape}")
print(f"\nMerged dataset columns: {merged_df.columns.tolist()}")
print(f"\nMissing values summary:")
print(merged_df.isnull().sum().head(15))

Original dataset shape: (901759, 9)
Cleaned dataset shape: (557347, 9)
Rows removed: 344412


KeyError: 'Id_accident'

In [5]:
# Merge accidents with other datasets
# First, let's prepare the data for merging

# Merge accidents with usagers (users/people involved)
merged_df = accidents_df_cleaned.merge(Usagers_df, on='Id_accident', how='left')

# Merge with vehicules
merged_df = merged_df.merge(vehicule_df, on='Id_accident', how='left')

# Merge with lieux (locations)
merged_df = merged_df.merge(Lieux_df, on='Id_accident', how='left')

print(f"Merged dataset shape: {merged_df.shape}")
print(f"\nMerged dataset columns: {merged_df.columns.tolist()}")
print(f"\nMissing values per column:")
print(merged_df.isnull().sum())

NameError: name 'accidents_df_cleaned' is not defined

## Feature Engineering and Preprocessing

In [ ]:
# Create a copy for feature engineering
df_model = merged_df.copy()

# Extract time features from 'hrmn' (hour and minute)
df_model['hour'] = (df_model['hrmn'] // 100).fillna(0).astype(int)
df_model['minute'] = (df_model['hrmn'] % 100).fillna(0).astype(int)

# Create time of day categories
def categorize_time(hour):
    if pd.isna(hour):
        return 'Unknown'
    hour = int(hour)
    if 6 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 18:
        return 'Afternoon'
    elif 18 <= hour < 24:
        return 'Evening'
    else:
        return 'Night'

df_model['time_of_day'] = df_model['hour'].apply(categorize_time)

# Create day type features
def categorize_day(jour):
    if pd.isna(jour):
        return 'Unknown'
    jour = int(jour)
    if jour in [6, 7]:  # Saturday and Sunday
        return 'Weekend'
    else:
        return 'Weekday'

df_model['day_type'] = df_model['jour'].apply(categorize_day)

# Create season features
def categorize_season(mois):
    if pd.isna(mois):
        return 'Unknown'
    mois = int(mois)
    if mois in [12, 1, 2]:
        return 'Winter'
    elif mois in [3, 4, 5]:
        return 'Spring'
    elif mois in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

df_model['season'] = df_model['mois'].apply(categorize_season)

# Create target variable: severity (0 = Light, 1 = Severe)
# Assuming 'Accident Léger' = Light, others = Severe
df_model['accident_severity'] = (df_model['Type Accident - Libellé'] != 'Accident Léger').astype(int)

print("Feature engineering completed!")
print(f"\nNew features created:")
print(f"- hour, minute")
print(f"- time_of_day: {df_model['time_of_day'].unique()}")
print(f"- day_type: {df_model['day_type'].unique()}")
print(f"- season: {df_model['season'].unique()}")
print(f"- accident_severity: {df_model['accident_severity'].value_counts().to_dict()}")
print(f"\nDataset shape: {df_model.shape}")

## Data Preparation for Modeling

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Select features for modeling
# Focus on features with reasonable data availability
feature_columns = [
    'Année', 'hour', 'minute', 'Age véhicule',
    'time_of_day', 'day_type', 'season',
    'Catégorie véhicule', 'Lieu Admin Actuel - Territoire Nom'
]

# Create modeling dataset
df_modeling = df_model[feature_columns + ['accident_severity']].copy()

# Drop rows with missing target
df_modeling = df_modeling.dropna(subset=['accident_severity'])

print(f"Dataset for modeling shape: {df_modeling.shape}")
print(f"Missing values:\n{df_modeling.isnull().sum()}")

# Handle missing values
df_modeling['Age véhicule'].fillna(df_modeling['Age véhicule'].median(), inplace=True)
df_modeling['hour'].fillna(0, inplace=True)
df_modeling['minute'].fillna(0, inplace=True)

# Encode categorical variables
label_encoders = {}
categorical_features = ['time_of_day', 'day_type', 'season', 'Catégorie véhicule', 'Lieu Admin Actuel - Territoire Nom']

for col in categorical_features:
    le = LabelEncoder()
    df_modeling[col] = le.fit_transform(df_modeling[col].astype(str))
    label_encoders[col] = le

print(f"\nCategorical features encoded!")
print(f"Final dataset shape: {df_modeling.shape}")
print(f"\nTarget distribution:")
print(df_modeling['accident_severity'].value_counts())
print(f"Target proportions:\n{df_modeling['accident_severity'].value_counts(normalize=True)}")

## Train-Test Split and Model Training

In [ ]:
# Prepare features and target
X = df_modeling.drop('accident_severity', axis=1)
y = df_modeling['accident_severity']

# Split data into train and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining set target distribution:")
print(y_train.value_counts())
print(f"\nTest set target distribution:")
print(y_test.value_counts())

In [ ]:
# Train multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Store results
results = {}

print("Training models...\n")

for model_name, model in models.items():
    print(f"Training {model_name}...")
    
    # Use scaled data for Logistic Regression, original for tree-based models
    if model_name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    auc_score = roc_auc_score(y_test, y_pred_proba)
    accuracy = (y_pred == y_test).mean()
    
    results[model_name] = {
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'accuracy': accuracy,
        'auc': auc_score
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  AUC-ROC: {auc_score:.4f}\n")

print("Model training completed!")

## Model Evaluation and Comparison

In [ ]:
# Detailed evaluation for each model
for model_name, result in results.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, result['y_pred'], 
                                target_names=['Light Accident', 'Severe Accident']))
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, result['y_pred']))

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Model Accuracy Comparison
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] for m in model_names]
aucs = [results[m]['auc'] for m in model_names]

ax = axes[0, 0]
x_pos = np.arange(len(model_names))
ax.bar(x_pos, accuracies, alpha=0.7, label='Accuracy')
ax.set_ylabel('Score')
ax.set_title('Model Accuracy Comparison')
ax.set_xticks(x_pos)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# 2. AUC-ROC Comparison
ax = axes[0, 1]
ax.bar(x_pos, aucs, alpha=0.7, color='orange', label='AUC-ROC')
ax.set_ylabel('Score')
ax.set_title('Model AUC-ROC Comparison')
ax.set_xticks(x_pos)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

# 3. ROC Curves
ax = axes[1, 0]
for model_name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['y_pred_proba'])
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={result['auc']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend()
ax.grid(alpha=0.3)

# 4. Confusion Matrix for best model
best_model_name = max(results.keys(), key=lambda x: results[x]['auc'])
best_result = results[best_model_name]
cm = confusion_matrix(y_test, best_result['y_pred'])

ax = axes[1, 1]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
            xticklabels=['Light', 'Severe'], yticklabels=['Light', 'Severe'])
ax.set_title(f'Confusion Matrix - {best_model_name}')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

print(f"\nBest Model: {best_model_name} with AUC-ROC = {best_result['auc']:.4f}")

## Feature Importance Analysis

In [ ]:
# Feature importance for tree-based models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Random Forest Feature Importance
rf_model = results['Random Forest']['model']
rf_importance = rf_model.feature_importances_
feature_names = X_train.columns

ax = axes[0]
sorted_idx = np.argsort(rf_importance)[-10:]  # Top 10 features
ax.barh(range(len(sorted_idx)), rf_importance[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Importance')
ax.set_title('Random Forest - Top 10 Feature Importance')
ax.grid(axis='x', alpha=0.3)

# Gradient Boosting Feature Importance
gb_model = results['Gradient Boosting']['model']
gb_importance = gb_model.feature_importances_

ax = axes[1]
sorted_idx = np.argsort(gb_importance)[-10:]  # Top 10 features
ax.barh(range(len(sorted_idx)), gb_importance[sorted_idx])
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel('Importance')
ax.set_title('Gradient Boosting - Top 10 Feature Importance')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print("Feature importance analysis completed!")

## Predictions on New Data

In [ ]:
# Example: Make predictions on new data
# Create a sample accident scenario
sample_accident = pd.DataFrame({
    'Année': [2024],
    'hour': [18],
    'minute': [30],
    'Age véhicule': [5],
    'time_of_day': ['Evening'],
    'day_type': ['Weekday'],
    'season': ['Summer'],
    'Catégorie véhicule': ['VT'],
    'Lieu Admin Actuel - Territoire Nom': ['Métropole']
})

# Encode categorical variables using the fitted encoders
sample_accident_encoded = sample_accident.copy()
for col in categorical_features:
    if col in label_encoders:
        sample_accident_encoded[col] = label_encoders[col].transform(sample_accident_encoded[col].astype(str))

# Make predictions with the best model
best_model = results[best_model_name]['model']

if best_model_name == 'Logistic Regression':
    sample_scaled = scaler.transform(sample_accident_encoded)
    prediction = best_model.predict(sample_scaled)[0]
    probability = best_model.predict_proba(sample_scaled)[0]
else:
    prediction = best_model.predict(sample_accident_encoded)[0]
    probability = best_model.predict_proba(sample_accident_encoded)[0]

print(f"Sample Accident Prediction ({best_model_name}):")
print(f"Input: {sample_accident.to_dict('records')[0]}")
print(f"\nPrediction: {'Severe Accident' if prediction == 1 else 'Light Accident'}")
print(f"Probability of Light Accident: {probability[0]:.2%}")
print(f"Probability of Severe Accident: {probability[1]:.2%}")

## Summary and Insights

In [ ]:
print("="*70)
print("ACCIDENT PREDICTION MODEL - SUMMARY")
print("="*70)

print("\n1. DATA OVERVIEW:")
print(f"   - Total records used: {len(df_modeling):,}")
print(f"   - Training set: {len(X_train):,} records")
print(f"   - Test set: {len(X_test):,} records")
print(f"   - Light accidents: {(y == 0).sum():,} ({(y == 0).sum()/len(y)*100:.1f}%)")
print(f"   - Severe accidents: {(y == 1).sum():,} ({(y == 1).sum()/len(y)*100:.1f}%)")

print("\n2. FEATURES USED:")
for i, feat in enumerate(X_train.columns, 1):
    print(f"   {i}. {feat}")

print("\n3. MODELS TRAINED:")
for model_name, result in results.items():
    print(f"   - {model_name}")
    print(f"     Accuracy: {result['accuracy']:.4f}")
    print(f"     AUC-ROC: {result['auc']:.4f}")

print(f"\n4. BEST MODEL: {best_model_name}")
print(f"   - AUC-ROC Score: {results[best_model_name]['auc']:.4f}")
print(f"   - Accuracy: {results[best_model_name]['accuracy']:.4f}")

print("\n5. KEY INSIGHTS:")
print("   - The model can predict accident severity based on temporal and vehicle features")
print("   - Time of day, day type, and season are important predictive factors")
print("   - Vehicle age and category also influence accident severity")
print("   - The model can be used for risk assessment and prevention strategies")

print("\n" + "="*70)

,Id_accident,Lettre Conventionnelle Véhicule,Année,Lieu Admin Actuel - Territoire Nom,Type Accident - Libellé,CNIT,Catégorie véhicule,Age véhicule,Type Accident - Libellé (old)
count,557347,557346,557347.000000,557347,557347,403646,557347,515516.000000,0
unique,339801,64,NaN,3,3,87433,9,NaN,0
top,770 390,A,NaN,Métropole,Accident Léger,LGG91C10G651,VT,NaN,NaN
freq,31,327095,NaN,532142,339509,1503,363571,NaN,NaN
mean,NaN,NaN,2017.391374,NaN,NaN,NaN,NaN,9.060328,NaN
std,NaN,NaN,1.663062,NaN,NaN,NaN,NaN,6.660152,NaN
min,NaN,NaN,2015.000000,NaN,NaN,NaN,NaN,0.000000,NaN
25%,NaN,NaN,2016.000000,NaN,NaN,NaN,NaN,4.000000,NaN
50%,NaN,NaN,2017.000000,NaN,NaN,NaN,NaN,8.000000,NaN
75%,NaN,NaN,2019.000000,NaN,NaN,NaN,NaN,14.000000,NaN
